# 16S Analyses of the Longitudinal Acne Study
## CTF and RPCA Plots

Date created: 10/18/2024

Notebook author: Britta De Pessemier 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- CTF plots 
- RPCA plots

In [314]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from adjustText import adjust_text

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact

# Gemelli RPCA and rarefaction utilities
from gemelli.rpca import rpca # Optional: import auto_rpca if you use auto_rpca later
from gemelli.utils import qc_rarefaction
from gemelli.preprocessing import rclr_transformation


In [84]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Calculate the angle between the x-axis and the largest eigenvector
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    
    # Width and height of the ellipse (scaling applied)
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    
    # Create the ellipse and add it to the plot
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

## CTF 
Analysis per participant skin site location C1/C2 for healthy and C1/C2/C3 for acne 


In [85]:

# Define the color palette for the groups
group_colors = {
    "Healthy": "#000096",  # Color for Healthy
    "Acne_L": "#ff676c",   # Color for Acne Lesional
    "Acne_NL": "#17c6d5"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
   # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD017.D14.C1  LAMI.RD003.D28.C2  LAMI.RD011.D0.C1  \
LAMI.RD017.D14.C1           0.000000          15.187849         13.257771   
LAMI.RD003.D28.C2          15.187849           0.000000          5.475794   
LAMI.RD011.D0.C1           13.257771           5.475794          0.000000   
LAMI.RD011.D28.C1          13.083151           5.925958          0.674414   
LAMI.RD006.D0.C2           13.581639           6.303939          0.955279   
...                              ...                ...               ...   
LAMI.RD310.D9.C1           10.333933           6.415692          5.613001   
LAMI.RD310.D21.C1          10.123376           6.583676          5.914402   
LAMI.RD310.D0.C1           10.588279           6.113444          5.464744   
LAMI.RD310.D14.C1          10.385060           6.272099          5.531075   
LAMI.RD310.D28.C1          10.042077           6.342499          5.498209   

                   LAMI.RD011.D28.C1  LAMI.RD006.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2523675088.py:107: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD006.D14.C2  LAMI.RD006.D14.C1  \
LAMI.RD011.D14.C1           0.000000           6.479502           7.139171   
LAMI.RD006.D14.C2           6.479502           0.000000           0.825376   
LAMI.RD006.D14.C1           7.139171           0.825376           0.000000   
LAMI.RD018.D0.C1            6.718586           5.770834           6.472288   
LAMI.RD018.D0.C2            6.491061           5.198449           5.919783   
...                              ...                ...                ...   
LAMI.RD314.D16.C1           5.805068           7.323252           7.994251   
LAMI.RD314.D21.C1           5.700178           7.377919           8.065342   
LAMI.RD314.D0.C1            6.335841           7.256653           7.877528   
LAMI.RD314.D28.C1           5.713597           7.539581           8.218511   
LAMI.RD314.D14.C1           5.861592           7.324871           7.996571   

                   LAMI.RD018.D0.C1  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2523675088.py:138: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## CTF colored by severity group

In [87]:
# Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values

    # Debugging: Check the shape of points
    print(f"Points shape for group '{group_name}': {points.shape}")

    if points.shape[0] < 2:  # Ensure at least 2 points for covariance
        print(f"Not enough points to compute covariance for group '{group_name}'")
        continue  # Correctly skip this group if not enough points

    # Calculate mean and covariance matrix for the group
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)

    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)


Points shape for group 'absent Acne_NL': (9, 2)
Points shape for group 'absent Healthy': (14, 2)
Points shape for group 'high Acne_L': (10, 2)
Points shape for group 'low Acne_L': (18, 2)
Points shape for group 'low Acne_NL': (9, 2)
Points shape for group 'moderate Acne_L': (23, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1708207561.py:21: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

In [284]:
# Define the color palette for the groups
group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_NL": "#b6ddea", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "absent Healthy": "absent Healthy",
    "absent Acne_NL": "absent Acne non-lesional",
    "low Acne_NL": "low Acne non-lesional", 
    "low Acne_L": "low Acne lesional",
    "moderate Acne_L": "moderate Acne lesional", 
    "high Acne_L": "high Acne lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata_severity_group.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["severity_group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        # Calculate points for the group
        points = group[["PC1", "PC2"]].values
        
        # Debugging: Check the shape of points
        print(f"Points shape for group '{group_name}': {points.shape}")

        # Check if the group has enough points for covariance calculation
        if points.shape[0] < 2:
            print(f"Not enough points to compute covariance for group '{group_name}'. Skipping this group.")
            continue  # Skip this group if not enough points

        # Create scatter plot for the group
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_severity_group.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD017.D14.C1  LAMI.RD003.D28.C2  LAMI.RD011.D0.C1  \
LAMI.RD017.D14.C1           0.000000          15.187849         13.257771   
LAMI.RD003.D28.C2          15.187849           0.000000          5.475794   
LAMI.RD011.D0.C1           13.257771           5.475794          0.000000   
LAMI.RD011.D28.C1          13.083151           5.925958          0.674414   
LAMI.RD006.D0.C2           13.581639           6.303939          0.955279   
...                              ...                ...               ...   
LAMI.RD310.D9.C1           10.333933           6.415692          5.613001   
LAMI.RD310.D21.C1          10.123376           6.583676          5.914402   
LAMI.RD310.D0.C1           10.588279           6.113444          5.464744   
LAMI.RD310.D14.C1          10.385060           6.272099          5.531075   
LAMI.RD310.D28.C1          10.042077           6.342499          5.498209   

                   LAMI.RD011.D28.C1  LAMI.RD006.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two 

Points shape for group 'high Acne_L': (1, 2)
Not enough points to compute covariance for group 'high Acne_L'. Skipping this group.
Points shape for group 'low Acne_L': (8, 2)
Points shape for group 'low Acne_NL': (3, 2)
Points shape for group 'moderate Acne_L': (11, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)


Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD006.D14.C2  LAMI.RD006.D14.C1  \
LAMI.RD011.D14.C1           0.000000           6.479502           7.139171   
LAMI.RD006.D14.C2           6.479502           0.000000           0.825376   
LAMI.RD006.D14.C1           7.139171           0.825376           0.000000   
LAMI.RD018.D0.C1            6.718586           5.770834           6.472288   
LAMI.RD018.D0.C2            6.491061           5.198449           5.919783   
...                              ...                ...                ...   
LAMI.RD314.D16.C1           5.805068           7.323252           7.994251   
LAMI.RD314.D21.C1           5.700178           7.377919           8.065342   
LAMI.RD314.D0.C1            6.335841           7.256653           7.877528   
LAMI.RD314.D28.C1           5.713597           7.539581           8.218511   
LAMI.RD314.D14.C1           5.861592           7.324871           7.996571   

                   LAMI.RD018.D0.C1  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two 

Points shape for group 'high Acne_L': (1, 2)
Not enough points to compute covariance for group 'high Acne_L'. Skipping this group.
Points shape for group 'low Acne_L': (8, 2)
Points shape for group 'low Acne_NL': (3, 2)
Points shape for group 'moderate Acne_L': (11, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/370074417.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [301]:
# Severity group 
metadata_df = pd.read_csv(f'../Metadata/metadata_final_22102024.tsv', sep='\t')

# Group by 'subject_ID_CC' and calculate the mean of 'local_lesion_severity'
mean_severity = metadata_df.groupby('subject_ID_CC')['local_lesion_severity'].transform('mean')

# Add this mean value as a new column in metadata_df
metadata_df['local_lesion_severity_mean_ID_CC'] = mean_severity

In [278]:
mf = metadata_df.groupby('subject_ID_CC').agg({'group':'first','local_lesion_severity_mean_ID_CC':'first'})

In [302]:
mf = metadata_df.groupby('subject_ID_CC').agg({'host_subject_id':'first','local_lesion_severity_mean_ID_CC':'first'})

In [297]:
print(mf)

              host_subject_id  local_lesion_severity_mean_ID_CC
subject_ID_CC                                                  
PP_11C1                 RD011                          0.000000
PP_11C2                 RD011                          0.000000
PP_14C1                 RD014                          0.000000
PP_14C2                 RD014                          0.000000
PP_17C1                 RD017                          0.000000
PP_17C2                 RD017                          0.000000
PP_18C1                 RD018                          0.000000
PP_18C2                 RD018                          0.000000
PP_1C1                  RD001                          0.000000
PP_1C2                  RD001                          0.000000
PP_2C1                  RD002                          0.000000
PP_2C2                  RD002                          0.000000
PP_304C1                RD304                          3.125000
PP_304C2                RD304           

In [303]:
mf.index.name = '#SampleID'

In [305]:
mf.to_csv('../Data/16S/CTF/ctf-results_174951/subject-metadata_subject_ID_mean_severity.tsv', sep='\t')

In [287]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from qiime2 import Artifact
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults

# Define the color gradient for 'local_lesion_severity_mean_ID_CC' as a colormap
custom_colors = ["#423fa6", "#67b2be", "#ededed", "#df7963", "#ca0000", "#960000"]
severity_cmap = LinearSegmentedColormap.from_list("severity_gradient", custom_colors)

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    distance_matrix = table.view(DistanceMatrix)
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    print(data)
    
    # Load the subject biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)
    
    # Extract sample coordinates and proportion explained
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # Load the metadata
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata_mean_severity.tsv', sep='\t')
    
    # Map the local lesion severity mean score for each subject
    spca_df["local_lesion_severity_mean_ID_CC"] = spca_df.index.map(metadata_df.set_index("#SampleID")["local_lesion_severity_mean_ID_CC"])
    
    # Rename columns for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'local_lesion_severity_mean_ID_CC']
    
    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Create a scatter plot with a gradient color based on 'local_lesion_severity_mean_ID_CC'
    sc = ax.scatter(
        spca_df['PC1'],
        spca_df['PC2'],
        c=spca_df['local_lesion_severity_mean_ID_CC'],
        cmap=severity_cmap,
        edgecolor='k',
        linewidths= 0.3,
        alpha=0.8,
        s=30  # Adjust size as needed
    )
    
    # Add color bar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Mean Local Lesion Severity per subjectID\'s C-zone ', fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Set simplified ticks
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add title
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Style the plot
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_local_lesion_severity_mean_ID_CC_continuous.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD017.D14.C1  LAMI.RD003.D28.C2  LAMI.RD011.D0.C1  \
LAMI.RD017.D14.C1           0.000000          15.187849         13.257771   
LAMI.RD003.D28.C2          15.187849           0.000000          5.475794   
LAMI.RD011.D0.C1           13.257771           5.475794          0.000000   
LAMI.RD011.D28.C1          13.083151           5.925958          0.674414   
LAMI.RD006.D0.C2           13.581639           6.303939          0.955279   
...                              ...                ...               ...   
LAMI.RD310.D9.C1           10.333933           6.415692          5.613001   
LAMI.RD310.D21.C1          10.123376           6.583676          5.914402   
LAMI.RD310.D0.C1           10.588279           6.113444          5.464744   
LAMI.RD310.D14.C1          10.385060           6.272099          5.531075   
LAMI.RD310.D28.C1          10.042077           6.342499          5.498209   

                   LAMI.RD011.D28.C1  LAMI.RD006.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1524760253.py:92: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD006.D14.C2  LAMI.RD006.D14.C1  \
LAMI.RD011.D14.C1           0.000000           6.479502           7.139171   
LAMI.RD006.D14.C2           6.479502           0.000000           0.825376   
LAMI.RD006.D14.C1           7.139171           0.825376           0.000000   
LAMI.RD018.D0.C1            6.718586           5.770834           6.472288   
LAMI.RD018.D0.C2            6.491061           5.198449           5.919783   
...                              ...                ...                ...   
LAMI.RD314.D16.C1           5.805068           7.323252           7.994251   
LAMI.RD314.D21.C1           5.700178           7.377919           8.065342   
LAMI.RD314.D0.C1            6.335841           7.256653           7.877528   
LAMI.RD314.D28.C1           5.713597           7.539581           8.218511   
LAMI.RD314.D14.C1           5.861592           7.324871           7.996571   

                   LAMI.RD018.D0.C1  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1524760253.py:92: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# CTF with severity scores that plot features 

In [293]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from qiime2 import Artifact
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from biom import load_table

# Define the color gradient for 'local_lesion_severity_mean_ID_CC' as a colormap
custom_colors = ["#423fa6", "#67b2be", "#ededed", "#df7963", "#ca0000", "#960000"]
severity_cmap = LinearSegmentedColormap.from_list("severity_gradient", custom_colors)

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    # Convert the biom table's data to a dense array and calculate feature prevalence
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]  # Convert to 1D array using .A1
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))


# Helper function to extract the last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    return ";".join(parts[-1:]) if "uncultured" not in taxon else parts[-3]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    distance_matrix = table.view(DistanceMatrix)
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Load the subject biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)
    
    # Extract sample coordinates and proportion explained
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # Load the metadata
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata_mean_severity.tsv', sep='\t')
    spca_df["local_lesion_severity_mean_ID_CC"] = spca_df.index.map(metadata_df.set_index("#SampleID")["local_lesion_severity_mean_ID_CC"])
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'local_lesion_severity_mean_ID_CC']
    
    # Load and filter the BIOM table by prevalence
    biom_path = f'/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Data/16S/RPCA/{folder}_rarefied_table_unannotated_absolute_abundances.biom'
    biom_table = load_table(biom_path)
    prevalence = calculate_prevalence(biom_table)
    prevalent_features = prevalence[prevalence >= 0.1].index  # Minimum 10% prevalence
    
    # Filter feature coordinates by prevalent features
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates = feature_coordinates.loc[feature_coordinates.index.isin(prevalent_features)]
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))
    
    # Select top 10 features by magnitude for plotting
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    top_feature_coords = feature_coordinates.loc[top_features]
    
    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Create a scatter plot with a gradient color based on 'local_lesion_severity_mean_ID_CC'
    sc = ax.scatter(
        spca_df['PC1'],
        spca_df['PC2'],
        c=spca_df['local_lesion_severity_mean_ID_CC'],
        cmap=severity_cmap,
        edgecolor='k',
        linewidths=0.3,
        alpha=0.8,
        s=30  # Adjust size as needed
    )
    
    # Add color bar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Mean Local Lesion Severity per subjectID\'s C-zone ', fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    
    # Plot feature arrows for top prevalent features
    for feature in top_feature_coords.index:
        ax.arrow(0, 0, top_feature_coords.loc[feature, 'PC1'], top_feature_coords.loc[feature, 'PC2'],
                 color='grey', alpha=0.3, head_width=0.02, length_includes_head=True)
        ax.text(top_feature_coords.loc[feature, 'PC1'] * 1.1,
                top_feature_coords.loc[feature, 'PC2'] * 1.1,
                top_feature_coords.loc[feature, 'Taxon'], fontsize=3, color='black', ha="center")
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Set simplified ticks
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add title
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Style the plot
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_local_lesion_severity_mean_ID_CC_continuous_with_features.png", dpi=600)
    plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1688137571.py:128: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1688137571.py:128: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [312]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from qiime2 import Artifact
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from biom import load_table

# Define the color gradient for 'local_lesion_severity_mean_ID_CC' as a colormap
custom_colors = ["#423fa6", "#67b2be", "#ededed", "#df7963", "#ca0000", "#960000"]
severity_cmap = LinearSegmentedColormap.from_list("severity_gradient", custom_colors)

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    # Convert the biom table's data to a dense array and calculate feature prevalence
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]  # Convert to 1D array using .A1
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))


# Helper function to extract the last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    return ";".join(parts[-1:]) if "uncultured" not in taxon else parts[-3]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    distance_matrix = table.view(DistanceMatrix)
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Load the subject biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)
    
    # Extract sample coordinates and proportion explained
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # Load the metadata
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata_subject_ID_mean_severity.tsv', sep='\t')
    spca_df["local_lesion_severity_mean_ID_CC"] = spca_df.index.map(metadata_df.set_index("#SampleID")["local_lesion_severity_mean_ID_CC"])
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'local_lesion_severity_mean_ID_CC']
    
    # Load and filter the BIOM table by prevalence
    biom_path = f'/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Data/16S/RPCA/{folder}_rarefied_table_unannotated_absolute_abundances.biom'
    biom_table = load_table(biom_path)
    prevalence = calculate_prevalence(biom_table)
    prevalent_features = prevalence[prevalence >= 0.1].index  # Minimum 10% prevalence
    
    # Filter feature coordinates by prevalent features
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates = feature_coordinates.loc[feature_coordinates.index.isin(prevalent_features)]
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))
    
    # Select top 10 features by magnitude for plotting
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    top_feature_coords = feature_coordinates.loc[top_features]
    
    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Create a scatter plot with a gradient color based on 'local_lesion_severity_mean_ID_CC'
    sc = ax.scatter(
        spca_df['PC1'],
        spca_df['PC2'],
        c=spca_df['local_lesion_severity_mean_ID_CC'],
        cmap=severity_cmap,
        edgecolor='k',
        linewidths=0.3,
        alpha=0.8,
        s=30  # Adjust size as needed
    )
    
    # Add color bar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Mean Local Lesion Severity per subjectID\'s C-zone ', fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    
    # Plot feature arrows for top prevalent features
    for feature in top_feature_coords.index:
        ax.arrow(0, 0, top_feature_coords.loc[feature, 'PC1'], top_feature_coords.loc[feature, 'PC2'],
                 color='grey', alpha=0.3, head_width=0.02, length_includes_head=True)
        ax.text(top_feature_coords.loc[feature, 'PC1'] * 1.1,
                top_feature_coords.loc[feature, 'PC2'] * 1.1,
                top_feature_coords.loc[feature, 'Taxon'], fontsize=3, color='black', ha="center")
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Set simplified ticks
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add title
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Style the plot
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_local_lesion_severity_mean_ID_CC_continuous_with_features.png", dpi=600)
    plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3111506893.py:128: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3111506893.py:128: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# CTF with severity scores that plot features and different shapes for the host_subject_id

# Rerun CTF for the severity groups 


In [89]:

# Define the color palette for the groups
group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_NL": "#b6ddea", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "absent Healthy": "absent Healthy",
    "absent Acne_NL": "absent Acne non-lesional",
    "low Acne_NL": "low Acne non-lesional", 
    "low Acne_L": "low Acne lesional",
    "moderate Acne_L": "moderate Acne lesional", 
    "high Acne_L": "high Acne lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_subject_ID_CC_LLS_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_subject_ID_CC_LLS_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_subject_ID_CC_LLS_{folder}/subject-metadata_subject_ID_CC_LLS.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["severity_group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        # Calculate points for the group
        points = group[["PC1", "PC2"]].values
        
        # Debugging: Check the shape of points
        print(f"Points shape for group '{group_name}': {points.shape}")

        # Check if the group has enough points for covariance calculation
        if points.shape[0] < 2:
            print(f"Not enough points to compute covariance for group '{group_name}'. Skipping this group.")
            continue  # Skip this group if not enough points

        # Create scatter plot for the group
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_subject_ID_CC_LLS_severity_group.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD304.D11.C2  LAMI.RD304.D28.C3  LAMI.RD305.D23.C2  \
LAMI.RD304.D11.C2           0.000000          12.477221           1.905424   
LAMI.RD304.D28.C3          12.477221           0.000000          12.676795   
LAMI.RD305.D23.C2           1.905424          12.676795           0.000000   
LAMI.RD317.D25.C1           7.319722           7.781132           7.031533   
LAMI.RD305.D28.C2          11.895487           5.032716          11.736082   
...                              ...                ...                ...   
LAMI.RD313.D9.C1            5.847184           6.974892           5.878797   
LAMI.RD313.D16.C1           5.985790           7.256817           5.794207   
LAMI.RD313.D0.C1            5.071833           7.522883           5.205771   
LAMI.RD313.D14.C1           5.076282           7.552796           5.191125   
LAMI.RD313.D28.C1           4.899011           7.841812           4.897679   

                   LAMI.RD317.D25.C1 

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

Points shape for group 'low Acne_NL': (9, 2)
Points shape for group 'moderate Acne_L': (23, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD308.D9.C3  LAMI.RD310.D18.C1  \
LAMI.RD011.D14.C1           0.000000         14.545560          10.672837   
LAMI.RD308.D9.C3           14.545560          0.000000           4.442995   
LAMI.RD310.D18.C1          10.672837          4.442995           0.000000   
LAMI.RD306.D28.C3          10.782609          5.333329           4.713322   
LAMI.RD306.D14.C1          15.085672          2.714641           5.585196   
...                              ...               ...                ...   
LAMI.RD313.D9.C1            1.718107         13.169930           9.183365   
LAMI.RD313.D16.C1           6.426703          8.352727           4.253405   
LAMI.RD313.D0.C1            3.568725         11.199180           7.227267   
LAMI.RD313.D28.C1           2.906889         11.769855           7.784092   
LAMI.RD313.D14.C1           4.026790         10.625794           6.672868   

                   LAMI.RD306.D28.C3  LAMI.RD306.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:124: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

Points shape for group 'low Acne_NL': (9, 2)
Points shape for group 'moderate Acne_L': (23, 2)


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2987052189.py:155: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# CTF without Acne_NL samples with score different than 0

In [250]:

# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject-metadata.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
   # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_filtered.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD017.D14.C1  LAMI.RD003.D28.C2  LAMI.RD319.D2.C3  \
LAMI.RD017.D14.C1           0.000000          21.536523          4.081905   
LAMI.RD003.D28.C2          21.536523           0.000000         24.977635   
LAMI.RD319.D2.C3            4.081905          24.977635          0.000000   
LAMI.RD011.D28.C1          12.606116          12.617917         14.997559   
LAMI.RD011.D0.C1           12.947615          13.018727         15.193759   
...                              ...                ...               ...   
LAMI.RD310.D9.C1           10.300314          15.959096         12.218312   
LAMI.RD310.D21.C1          10.096202          15.714521         12.127897   
LAMI.RD310.D28.C1          10.238556          15.828910         12.179136   
LAMI.RD310.D14.C1          10.415101          16.037639         12.269049   
LAMI.RD310.D0.C1           10.742818          16.303830         12.484702   

                   LAMI.RD011.D28.C1  LAMI.RD011.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1129573241.py:107: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two

Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD006.D14.C2  LAMI.RD006.D14.C1  \
LAMI.RD011.D14.C1           0.000000           7.439045           5.694700   
LAMI.RD006.D14.C2           7.439045           0.000000           5.552700   
LAMI.RD006.D14.C1           5.694700           5.552700           0.000000   
LAMI.RD018.D0.C1            6.046601           6.071910           2.439770   
LAMI.RD018.D0.C2            6.535909           7.240319           2.760986   
...                              ...                ...                ...   
LAMI.RD304.D11.C1           5.281908           5.107300           5.808198   
LAMI.RD304.D16.C1           5.241564           5.757897           6.026550   
LAMI.RD304.D28.C1           6.590364           7.164908           8.080555   
LAMI.RD304.D0.C1            5.536302           5.025774           6.447380   
LAMI.RD304.D14.C1           6.491766           7.075496           7.533146   

                   LAMI.RD018.D0.C1  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1129573241.py:107: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1129573241.py:138: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [218]:
# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]  # Provide more context if 'uncultured'
    else:
        return parts[-1]  # Use only the last part if no 'uncultured'


In [206]:
# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    
    # Check if "uncultured" is present in the taxon
    if "uncultured" in taxon:
        # Extract the last three parts for more context if "uncultured" is present
        #return ";".join(parts[-3:])
        return parts[-3]
    else:
        # Otherwise, return the last part
        return ";".join(parts[-1:])


# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Rename the feature coordinates columns (0 -> 'PC1', 1 -> 'PC2', 2 -> 'PC3')
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject-metadata.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Load the taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to the feature_coordinates
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude (PC1^2 + PC2^2) and select top 10 features
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    top_feature_coords = feature_coordinates.loc[top_features]
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Plot feature arrows for the top 10 features
    for feature in top_feature_coords.index:
        ax.arrow(0, 0, top_feature_coords.loc[feature, 'PC1'], top_feature_coords.loc[feature, 'PC2'],
                 color='grey', alpha=0.3, head_width=0.02, length_includes_head=True)
        ax.text(top_feature_coords.loc[feature, 'PC1'] * 1.1,
                top_feature_coords.loc[feature, 'PC2'] * 1.1,
                feature_coordinates.loc[feature, 'Taxon'], fontsize=5, color='black')
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_filtered.png", dpi=600)
    plt.show()



Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2889269031.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3809550203.py:119: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2889269031.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only t

Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2889269031.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3809550203.py:119: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2889269031.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only t

# CTF without Acne_L different than 0 and visualized features have a prevalence higher than 10%

In [ ]:
# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    # Convert the biom table's data to a dense array and calculate feature prevalence
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]  # Convert to 1D array using .A1
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]  # Provide more context if 'uncultured'
    else:
        return parts[-1]  # Use only the last part if no 'uncultured'


In [313]:
# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    
    # Check if "uncultured" is present in the taxon
    if "uncultured" in taxon:
        # Extract the last three parts for more context if "uncultured" is present
        #return ";".join(parts[-3:])
        return parts[-3]
    else:
        # Otherwise, return the last part
        return ";".join(parts[-1:])


# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
   
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject-metadata.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Load the taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

        
    # Load and filter the BIOM table by prevalence
    biom_path = f'/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Data/16S/RPCA/{folder}_rarefied_table_unannotated_absolute_abundances.biom'
    biom_table = load_table(biom_path)
    prevalence = calculate_prevalence(biom_table)
    prevalent_features = prevalence[prevalence >= 0.1].index  # Minimum 10% prevalence
    
    # Filter feature coordinates by prevalent features
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates = feature_coordinates.loc[feature_coordinates.index.isin(prevalent_features)]
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))
    
    # Select top 10 features by magnitude for plotting
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    top_feature_coords = feature_coordinates.loc[top_features]
    
    
    # Sort feature coordinates by magnitude (PC1^2 + PC2^2) and select top 10 features
    #top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    #top_feature_coords = feature_coordinates.loc[top_features]
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Plot feature arrows for the top 10 features
    for feature in top_feature_coords.index:
        ax.arrow(0, 0, top_feature_coords.loc[feature, 'PC1'], top_feature_coords.loc[feature, 'PC2'],
                 color='grey', alpha=0.3, head_width=0.02, length_includes_head=True)
        ax.text(top_feature_coords.loc[feature, 'PC1'] * 1.1,
                top_feature_coords.loc[feature, 'PC2'] * 1.1,
                feature_coordinates.loc[feature, 'Taxon'], fontsize=5, color='black')
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}_prevalence_filtered.png", dpi=600)
    plt.show()



Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/934714503.py:140: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two 

Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/934714503.py:140: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two 

# Arcsine transformed

In [269]:
from qiime2 import Artifact
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# Define the color palette for the groups
group_colors = {
    "Healthy": "#000096",  # Color for Healthy
    "Acne_L": "#ff676c",   # Color for Acne Lesional
    "Acne_NL": "#17c6d5"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    
    # Check if "uncultured" is present in the taxon
    if "uncultured" in taxon:
        return parts[-3]  # Return the third-to-last part if uncultured
    else:
        return parts[-1]  # Otherwise, return the last part

# Function to calculate relative abundance
def calculate_relative_abundance(df):
    return df.div(df.sum(axis=1), axis=0)

# Function to apply arcsine square root transformation
def arcsine_transform(df):
    return np.arcsin(np.sqrt(df))

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/distance_matrix.qza')
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    # Normalize the absolute abundance to relative abundance
    relative_abundance_df = calculate_relative_abundance(feature_abundance_df)
    # Add a small pseudocount (if necessary) to avoid issues with zeros
    relative_abundance_df += 1e-6

    # Apply arcsine square root transformation to the relative abundance data
    arcsine_transformed_df = arcsine_transform(relative_abundance_df)
    
    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')


    # Align metadata with feature data (samples are in the rows)
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])

    # Check if common samples exist
    if len(common_samples) > 0:
    # Subset both DataFrames to only keep common samples
        arcsine_transformed_df = arcsine_transformed_df.loc[common_samples, :]
        metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
    
    # Assign 'Group' from metadata to arcsine_transformed_df
        arcsine_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']
    else:
        print("No common samples found. Check for mismatches in sample IDs.")


    # Load the subject_biplot artifact to get the feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract the feature coordinates
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load the taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to the feature_coordinates
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude (PC1^2 + PC2^2) and select top 10 features
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index

    # For each top feature, create a boxplot of arcsine-transformed relative abundance across groups
    for feature in top_features:
        # Create a boxplot for the arcsine-transformed relative abundance per group
        plt.figure(figsize=(8, 6))
        
        # Merge the arcsine-transformed feature data with the metadata
        feature_data = arcsine_transformed_df[[feature, 'Group']].copy()
        
        # Plot the arcsine-transformed relative abundance of the feature across the three groups
        ax = sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors)
        
        # Add the strip plot for better visualization
        sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
        
        # Add the title and labels
        plt.title(f'Arcsine Transformed Relative Abundance of {feature_coordinates.loc[feature, "Taxon"]}', fontsize=18)
        plt.xlabel('')
        plt.ylabel('Arcsine Relative Abundance', fontsize=16)
        
        # Custom x-axis labels
        new_labels = ['Acne\nLesional', 'Acne\nNon-lesional', 'Healthy']
        plt.xticks(ticks=range(len(new_labels)), labels=new_labels, rotation=0, ha='center', fontsize=16)
        
        # Save the plot
        plt.tight_layout()
        plt.savefig(f'../Figures/16S_Figures/{folder}_feature_{feature}_arcsine_boxplot.png', dpi=600)
        plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:113: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:127: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:113: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:127: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0

Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:113: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:127: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:113: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1816769217.py:127: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0

In [192]:
from qiime2 import Artifact
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# Define the color palette for the groups
group_colors = {
    "Healthy": "#000096",  # Color for Healthy
    "Acne_L": "#ff676c",   # Color for Acne Lesional
    "Acne_NL": "#17c6d5"   # Color for Acne Non-lesional
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    
    # Check if "uncultured" is present in the taxon
    if "uncultured" in taxon:
        return parts[-3]  # Return the third-to-last part if uncultured
    else:
        return parts[-1]  # Otherwise, return the last part

# Function to calculate relative abundance
def calculate_relative_abundance(df):
    return df.div(df.sum(axis=1), axis=0)

# Function to apply arcsine square root transformation
def arcsine_transform(df):
    return np.arcsin(np.sqrt(df))

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    # Normalize the absolute abundance to relative abundance
    relative_abundance_df = calculate_relative_abundance(feature_abundance_df)
    # Add a small pseudocount to avoid zeros
    relative_abundance_df += 1e-6

    # Apply arcsine square root transformation
    arcsine_transformed_df = arcsine_transform(relative_abundance_df)

    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data (samples are in the rows)
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])
    arcsine_transformed_df = arcsine_transformed_df.loc[common_samples, :]
    metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
    arcsine_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']

    # Load the subject_biplot artifact to get feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract feature coordinates
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add taxon information
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude (PC1^2 + PC2^2) and select top 10 features
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index

    # Initialize a plot for all top features
    plt.figure(figsize=(10, 6 * len(top_features)))  # Adjust height based on the number of top features

    # Loop over top features to create a boxplot for each
    for idx, feature in enumerate(top_features):
        plt.subplot(len(top_features), 1, idx + 1)  # Create a subplot for each feature
        
        # Merge feature data with metadata
        feature_data = arcsine_transformed_df[[feature, 'Group']].copy()
        
        # Plot the arcsine-transformed relative abundance of the feature across the groups
        ax = sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors)
        sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)

        # Set y-axis label as the Taxon name
        ax.set_ylabel(f'Arcsine Transformed Relative Abundance ({feature_coordinates.loc[feature, "Taxon"]})', fontsize=12)
        ax.set_xlabel('')

        # Mann-Whitney U test for significance between groups
        groups = ['Healthy', 'Acne_L', 'Acne_NL']
        p_values = {}

        y_max = feature_data[feature].max() + 0.05  # Set height for significance annotations
        height_step = 0.05  # Vertical distance between significance bars
        
        # Perform pairwise comparisons
        for i, group1 in enumerate(groups):
            for j, group2 in enumerate(groups):
                if i < j:
                    # Get values for both groups
                    group1_values = feature_data[feature_data['Group'] == group1][feature]
                    group2_values = feature_data[feature_data['Group'] == group2][feature]

                    # Perform Mann-Whitney U test
                    stat, p = mannwhitneyu(group1_values, group2_values, alternative='two-sided')
                    p_values[f'{group1} vs {group2}'] = p

                    # Determine significance label
                    if p >= 0.05:
                        label = 'ns'
                    elif p < 0.001:
                        label = '***'
                    elif p < 0.01:
                        label = '**'
                    else:
                        label = '*'

                    # Add significance bars and labels
                    x1, x2 = i, j
                    y = y_max
                    plt.plot([x1, x1, x2, x2], [y, y + 0.01, y + 0.01, y], lw=1, color='black')
                    plt.text((x1 + x2) * 0.5, y + 0.02, label, ha='center', va='bottom', fontsize=10)

                    # Update y_max for the next comparison
                    y_max += height_step

    plt.tight_layout()
    plt.savefig(f'../Figures/16S_Figures/{folder}_top_features_arcsine_boxplots.png', dpi=600)
    plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.

Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3953754402.py:93: FutureWarning: Passing `palette` without assigning `hue` is deprecated.

In [195]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
from qiime2 import Artifact

# Define the color palette for the groups
group_colors = {
    "Healthy": "#000096",  # Color for Healthy
    "Acne_L": "#ff676c",   # Color for Acne Lesional
    "Acne_NL": "#17c6d5"   # Color for Acne Non-lesional
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    
    # Check if "uncultured" is present in the taxon
    if "uncultured" in taxon:
        return parts[-3]  # Return the third-to-last part if uncultured
    else:
        return parts[-1]  # Otherwise, return the last part

# Function to calculate relative abundance
def calculate_relative_abundance(df):
    return df.div(df.sum(axis=1), axis=0)

# Function to apply arcsine square root transformation
def arcsine_transform(df):
    return np.arcsin(np.sqrt(df))

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    # Normalize the absolute abundance to relative abundance
    relative_abundance_df = calculate_relative_abundance(feature_abundance_df)
    # Add a small pseudocount (if necessary) to avoid issues with zeros
    relative_abundance_df += 1e-6

    # Apply arcsine square root transformation to the relative abundance data
    arcsine_transformed_df = arcsine_transform(relative_abundance_df)
    
    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data (samples are in the rows)
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])

    # Check if common samples exist
    if len(common_samples) > 0:
        arcsine_transformed_df = arcsine_transformed_df.loc[common_samples, :]
        metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
        arcsine_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']
    else:
        print("No common samples found. Check for mismatches in sample IDs.")

    # Load the subject_biplot artifact to get the feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract the feature coordinates
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load the taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to the feature_coordinates
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude (PC1^2 + PC2^2) and select top 9 features (3 plots per row)
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(9).index

    # Create a figure with 3 rows of subplots
    fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)

    # Flatten the axes array for easier indexing
    axes = axes.flatten()

    # For each top feature, create a boxplot of arcsine-transformed relative abundance across groups
    for i, feature in enumerate(top_features):
        # Get the axis for this plot
        ax = axes[i]

        # Merge the arcsine-transformed feature data with the metadata
        feature_data = arcsine_transformed_df[[feature, 'Group']].copy()

        # Plot the arcsine-transformed relative abundance of the feature across the three groups
        sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors, ax=ax)
        
        # Add the strip plot for better visualization
        sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
        
        # Set the y-axis label to the taxon name
        ax.set_ylabel(f'Arcsine Transformed\n{feature_coordinates.loc[feature, "Taxon"]}', fontsize=12)
        
        # Remove x-axis labels (shared across plots)
        ax.set_xlabel('')
        
        # Set custom x-axis labels for each plot
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(['Acne Lesional', 'Acne Non-lesional', 'Healthy'], rotation=0, ha='center')

    # Adjust layout and save the figure
    plt.tight_layout()
    plt.savefig(f'../Figures/16S_Figures/{file_suffix}_top_features_arcsine_boxplots.png', dpi=300)
    plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is depreca

Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1858221496.py:104: FutureWarning: Passing `palette` without assigning `hue` is depreca

In [205]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
from qiime2 import Artifact

# Define the color palette for the groups
group_colors = {
    "Healthy": "#000096",  # Color for Healthy
    "Acne_L": "#ff676c",   # Color for Acne Lesional
    "Acne_NL": "#17c6d5"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}
# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    
    # Check if "uncultured" is present in the taxon
    if "uncultured" in taxon:
        return parts[-3]  # Return the third-to-last part if uncultured
    else:
        return parts[-1]  # Otherwise, return the last part

# Function to calculate relative abundance
def calculate_relative_abundance(df):
    return df.div(df.sum(axis=1), axis=0)

# Function to apply arcsine square root transformation
def arcsine_transform(df):
    return np.arcsin(np.sqrt(df))

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    # Normalize the absolute abundance to relative abundance
    relative_abundance_df = calculate_relative_abundance(feature_abundance_df)
    # Add a small pseudocount (if necessary) to avoid issues with zeros
    relative_abundance_df += 1e-6

    # Apply arcsine square root transformation to the relative abundance data
    arcsine_transformed_df = arcsine_transform(relative_abundance_df)
    
    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data (samples are in the rows)
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])

    if len(common_samples) > 0:
        arcsine_transformed_df = arcsine_transformed_df.loc[common_samples, :]
        metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
        arcsine_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']
        #arcsine_transformed_df['Group'] = arcsine_transformed_df['Group'].map(group_name_mapping)

    else:
        print("No common samples found. Check for mismatches in sample IDs.")

    # Load the subject_biplot artifact to get the feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract the feature coordinates
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load the taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to the feature_coordinates
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude (PC1^2 + PC2^2) and select top 9 features (3 plots per row)
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(9).index

    # Create a figure with 3 rows of subplots
    fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True, sharey=True)

    # Flatten the axes array for easier indexing
    axes = axes.flatten()

    # List of groups
    groups = ['Acne_L', 'Acne_NL', 'Healthy']
    
    # Loop through top features and create the plots
    for i, feature in enumerate(top_features):
        # Get the axis for this plot
        ax = axes[i]

        # Merge the arcsine-transformed feature data with the metadata
        feature_data = arcsine_transformed_df[[feature, 'Group']].copy()

        # Plot the arcsine-transformed relative abundance of the feature across the three groups
        sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors, ax=ax, width=0.5)
        
        # Add the strip plot for better visualization
        sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
        
        # Set the y-axis label to the taxon name
        ax.set_ylabel(f'Arcsine Transformed\n{feature_coordinates.loc[feature, "Taxon"]}', fontsize=12)
        
        # Set custom x-axis labels for each plot
        ax.set_xticks([0, 1, 2])
        #ax.set_xticklabels(rotation=0, ha='center')

        # Perform pairwise Mann-Whitney U tests and add significance
        y_max = feature_data[feature].max() * 1.1  # Set y_max to place significance lines
        height_step = 0.05
        
        for j, group1 in enumerate(groups):
            for k, group2 in enumerate(groups):
                if j < k:
                    # Extract group values
                    group1_values = feature_data[feature_data['Group'] == group1][feature]
                    group2_values = feature_data[feature_data['Group'] == group2][feature]

                    # Perform the Mann-Whitney U test
                    stat, p = mannwhitneyu(group1_values, group2_values, alternative='two-sided')

                    # Determine the significance label based on p-value thresholds
                    if p >= 0.05:
                        label = 'ns'
                    elif p < 0.001:
                        label = '***'
                    elif p < 0.01:
                        label = '**'
                    else:
                        label = '*'

                    # Get x coordinates of the boxplots
                    x1, x2 = j, k
                    y = y_max + height_step  # Vertical position for the horizontal line

                    # Draw horizontal line and annotate the significance label
                    ax.plot([x1, x1, x2, x2], [y, y + 0.01, y + 0.01, y], lw=1, color='black')
                    ax.text((x1 + x2) * 0.5, y + 0.02, label, ha='center', va='bottom', fontsize=12)

                    # Update y_max for the next comparison
                    y_max += height_step + 0.05

    # Adjust layout and save the figure
    plt.tight_layout()
    plt.savefig(f'../Figures/16S_Figures/{file_suffix}_top_features_arcsine_boxplots_with_significance.png', dpi=600)
    plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is depreca

Processing folder: 174951


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1253406922.py:114: FutureWarning: Passing `palette` without assigning `hue` is depreca

In [173]:
# Check if top features are in the abundance table by printing the IDs
print("Checking feature IDs in ordination vs. abundance table...")

# Clean and standardize feature IDs (if needed)
arcsine_transformed_df.columns = arcsine_transformed_df.columns.str.strip()
top_features_cleaned = top_features.str.strip()  # Strip any extra spaces or characters from top features

# Compare feature sets
print("Feature IDs from ordination (top features):")
print(top_features_cleaned)

print("Feature IDs from arcsine-transformed abundance table:")
print(arcsine_transformed_df.columns)

# Filter top features to only those that exist in the arcsine_transformed_df columns
top_features_filtered = [feature for feature in top_features_cleaned if feature in arcsine_transformed_df.columns]

# Print warning for missing features
missing_features = set(top_features_cleaned) - set(top_features_filtered)
if missing_features:
    print(f"Warning: The following top features were not found in the abundance table: {missing_features}")



Checking feature IDs in ordination vs. abundance table...
Feature IDs from ordination (top features):
Index(['TACGTAGGTGGCAAGCGTTATCCGGAATTATTGGGCGTAAAGCGCGCGTAGGCGGTTTTTTAAGTCTGATGTGAAAGCCCACGGCTCAACCGTGGAGGGTCATTGGAAACTGGAAAACTTGAGTGCAGAAGAGGAAAGTGGAATTCCATG',
       'TACGTAGGGTGCAAGCGTTAATCGGAATTATTGGGCGTAAAGCGAGTGCAGACGGTTACTTAAGCCAGATGTGAAATCCCCAAGCTTAACTTGGGACGTGCATTTGGAACTGGGTGACTAGAGTGTGTCAGAGGGAGGTAGAATTCCACA',
       'TACAGAGGGTGCGAGCGTTAATCGGAATTACTGGGCGTAAAGCGAGTGTAGGTGGCTCATTAAGTCACATGTGAAATCCCCGGGCTTAACCTGGGAACTGCATGTGATACTGGTGGTGCTAGAATATGTGAGAGGGAAGTAGAATTCCAG',
       'TACAGAGGGTGCAAGCGTTAATCGGATTTACTGGGCGTAAAGCGCGCGTAGGTGGCCAATTAAGTCAAATGTGAAATCCCCGAGCTTAACTTGGGAATTGCATTCGATACTGGTTGGCTAGAGTATGGGAGAGGATGGTAGAATTCCAGG',
       'TACGGAGGGTGCGAGCGTTAATCGGAATAACTGGGCGTAAAGGGCACGCAGGCGGTGACTTAAGTGAGGTGTGAAAGCCCCGGGCTTAACCTGGGAATTGCATTTCATACTGGGTCGCTAGAGTACTTTAGGGAGGGGTAGAATTCCACG',
       'TACGTAGGGGGCAAGCGTTATCCGGATTTACTGGGTGTAAAGGGAGCGTAGACGGAGCAGCAAGTCTGATGTGAAAGGCGGGGGC

In [169]:
print("Available features in arcsine_transformed_df:")
print(arcsine_transformed_df.columns)


Available features in arcsine_transformed_df:
Index(['GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT',
       'AGCGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAGCGCCGTAGCAATACGGAGCGGCAGACGGGTGAGTAACACGTGGGAACGTACCTTTTGGTTCGGAACAACACAGGGAAACTTGTGCTAATACCGGATAAGCCCTTACGGGGA',
       'ATTGAACGCTGGCGGCATGCCTTACACATGCAAGTCGAACGGTAACAGGTCTTCGGATGCTGACGAGTGGCGAACGGGTGAGTAATACATCGGAACGTGCCCGATCGTGGGGGATAACGGAGCGAAAGCTTTGCTAATACCGCATACGAT',
       'GATGAACGCTAGCGATAGGCTTAACACATGCAAGTCGAGGGGCAGCATGGTCTTAGCTTGCTAAGACTGATGGCGACCGGCGCACGGGTGCGTAACGCGTATGCAACTTGCCTCACAGAGGGGGATAACCCGTCGAAAGACGGACTAATA',
       'GACGAACGCTGGCGGCGCGCCTAACACATGCAAGTCGAACGGAGCTTAGAGAGCTTGCTTTTTAAGCTTAGTGGCGAACGGGTGAGTAACGCGTGAGTAACCTGCCCTAGAGTGGGGGACAACAGTTGGAAACGACTGCTAATACCGCAT',
       'ATTGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAACGATGATTATCTAGCTTGCTAGATATGATTAGTGGCGGACGGGTGAGTAACATTTAGGAATCTGCCTAGTAGTGGGGGATAGCTCGGGGAAACTCGAATTAA

In [170]:
print(top_features)

['TACGTAGGTGGCAAGCGTTATCCGGAATTATTGGGCGTAAAGCGCGCGTAGGCGGTTTTTTAAGTCTGATGTGAAAGCCCACGGCTCAACCGTGGAGGGTCATTGGAAACTGGAAAACTTGAGTGCAGAAGAGGAAAGTGGAATTCCATG', 'TACGTAGGGTGCAAGCGTTAATCGGAATTATTGGGCGTAAAGCGAGTGCAGACGGTTACTTAAGCCAGATGTGAAATCCCCAAGCTTAACTTGGGACGTGCATTTGGAACTGGGTGACTAGAGTGTGTCAGAGGGAGGTAGAATTCCACA', 'TACAGAGGGTGCGAGCGTTAATCGGAATTACTGGGCGTAAAGCGAGTGTAGGTGGCTCATTAAGTCACATGTGAAATCCCCGGGCTTAACCTGGGAACTGCATGTGATACTGGTGGTGCTAGAATATGTGAGAGGGAAGTAGAATTCCAG', 'TACAGAGGGTGCAAGCGTTAATCGGATTTACTGGGCGTAAAGCGCGCGTAGGTGGCCAATTAAGTCAAATGTGAAATCCCCGAGCTTAACTTGGGAATTGCATTCGATACTGGTTGGCTAGAGTATGGGAGAGGATGGTAGAATTCCAGG', 'TACGGAGGGTGCGAGCGTTAATCGGAATAACTGGGCGTAAAGGGCACGCAGGCGGTGACTTAAGTGAGGTGTGAAAGCCCCGGGCTTAACCTGGGAATTGCATTTCATACTGGGTCGCTAGAGTACTTTAGGGAGGGGTAGAATTCCACG', 'TACGTAGGGGGCAAGCGTTATCCGGATTTACTGGGTGTAAAGGGAGCGTAGACGGAGCAGCAAGTCTGATGTGAAAGGCGGGGGCTCAACCCCCGGACTGCATTGGAAACTGTTGATCTTGAGTACCGGAGAGGTAAGCGGAATTCCTAG', 'TACGTAGGGCGCGAGCGTTGTCCGGAATTATTGGGCGTAAAGGGCTTGTAGGCGGCTTGTCGCGTCTGCCGTGA

In [259]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
from qiime2 import Artifact

# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]
    else:
        return parts[-1]

# Function to calculate relative abundance
def calculate_relative_abundance(df):
    return df.div(df.sum(axis=1), axis=0)

# Function to apply arcsine square root transformation
def arcsine_transform(df):
    return np.arcsin(np.sqrt(df))

# Function to add significance brackets manually
def add_significance_bracket(ax, start, end, height, text, linewidth=1.0):
    ax.plot([start, end], [height, height], color='black', lw=linewidth)
    ax.text((start + end) * 0.5, height, text, ha='center', va='bottom', fontsize=12)

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    # Normalize the absolute abundance to relative abundance
    relative_abundance_df = calculate_relative_abundance(feature_abundance_df)
    relative_abundance_df += 1e-6  # Add pseudocount to avoid zeros

    # Apply arcsine square root transformation
    arcsine_transformed_df = arcsine_transform(relative_abundance_df)
    
    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])
    arcsine_transformed_df = arcsine_transformed_df.loc[common_samples, :]
    metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
    arcsine_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']

    # Set the explicit order of the Group column
    arcsine_transformed_df['Group'] = pd.Categorical(arcsine_transformed_df['Group'], categories=['Healthy', 'Acne_NL', 'Acne_L'], ordered=True)

    # Load the subject_biplot artifact to get feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract the feature coordinates
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to feature coordinates
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude and select top 9 features
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(9).index

    # Create a figure with 3 rows of subplots
    fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)
    axes = axes.flatten()

    # For each top feature, create a boxplot
    for i, feature in enumerate(top_features):
        ax = axes[i]

        # Merge feature data with metadata
        feature_data = arcsine_transformed_df[[feature, 'Group']].copy()

        # Plot the arcsine-transformed relative abundance
        sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors, ax=ax)
        sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
        
        # Set the y-axis label to the taxon name
        ax.set_ylabel(f'Arcsine Transformed\n{feature_coordinates.loc[feature, "Taxon"]}', fontsize=12)

        # Perform Mann-Whitney U test for significance
        # Ensure group_labels match the actual order in the plot (Healthy, Acne_NL, Acne_L)
        group_labels = ['Healthy', 'Acne_NL', 'Acne_L']
        pairs = [(0, 1), (0, 2), (1, 2)]
        p_values = []
        print(f"P-values for feature: {feature_coordinates.loc[feature, 'Taxon']}")
        for pair in pairs:
            group1 = group_labels[pair[0]]
            group2 = group_labels[pair[1]]
            data1 = feature_data[feature_data['Group'] == group1][feature]
            data2 = feature_data[feature_data['Group'] == group2][feature]
            _, p_value = mannwhitneyu(data1, data2)
            p_values.append(p_value)
            print(f"  {group1} vs {group2}: p-value = {p_value:.4f}")

        # Set height for significance brackets
        max_height = feature_data[feature].max() * 1.1
        for j, (start, end) in enumerate(pairs):
            height = max_height * (1 + j * 0.1)  # Adjust the height for each bracket
            significance = "***" if p_values[j] < 0.001 else "**" if p_values[j] < 0.01 else "*" if p_values[j] < 0.05 else "ns"
            add_significance_bracket(ax, start, end, height, significance)

    plt.tight_layout()
    plt.savefig(f'../Figures/16S_Figures/{file_suffix}_top_features_arcsine_boxplots.png', dpi=300)
    plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)


P-values for feature:  f__Caulobacteraceae
  Healthy vs Acne_NL: p-value = 0.4745
  Healthy vs Acne_L: p-value = 0.0844
  Acne_NL vs Acne_L: p-value = 1.0000
P-values for feature:  g__Cutibacterium
  Healthy vs Acne_NL: p-value = 0.0340
  Healthy vs Acne_L: p-value = 0.0271
  Acne_NL vs Acne_L: p-value = 0.3961
P-values for feature:  g__Mycobacterium
  Healthy vs Acne_NL: p-value = 0.0413
  Healthy vs Acne_L: p-value = 0.0000
  Acne_NL vs Acne_L: p-value = 0.6807


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)


P-values for feature:  g__Methylotenera
  Healthy vs Acne_NL: p-value = 0.2989
  Healthy vs Acne_L: p-value = 0.0140
  Acne_NL vs Acne_L: p-value = 1.0000
P-values for feature:  g__Phenylobacterium
  Healthy vs Acne_NL: p-value = 0.4745
  Healthy vs Acne_L: p-value = 0.4146
  Acne_NL vs Acne_L: p-value = 0.6807
P-values for feature:  s__Lactobacillus_iners
  Healthy vs Acne_NL: p-value = 0.0825
  Healthy vs Acne_L: p-value = 0.0012
  Acne_NL vs Acne_L: p-value = 0.7147


...

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.

P-values for feature:  g__Staphylococcus
  Healthy vs Acne_NL: p-value = 0.7489
  Healthy vs Acne_L: p-value = 0.8104
  Acne_NL vs Acne_L: p-value = 0.6373
P-values for feature:  f__Neisseriaceae
  Healthy vs Acne_NL: p-value = 0.0002
  Healthy vs Acne_L: p-value = 0.0000
  Acne_NL vs Acne_L: p-value = 0.5472
P-values for feature:  g__Enhydrobacter
  Healthy vs Acne_NL: p-value = 0.0000
  Healthy vs Acne_L: p-value = 0.0000
  Acne_NL vs Acne_L: p-value = 0.8570
P-values for feature:  g__Acinetobacter
  Healthy vs Acne_NL: p-value = 0.8008
  Healthy vs Acne_L: p-value = 0.0288
  Acne_NL vs Acne_L: p-value = 0.1882
P-values for feature:  g__Haemophilus
  Healthy vs Acne_NL: p-value = 0.0052
  Healthy vs Acne_L: p-value = 0.0004
  Acne_NL vs Acne_L: p-value = 0.8133
P-values for feature:  s__Blautia_caecimuris
  Healthy vs Acne_NL: p-value = 0.3356
  Healthy vs Acne_L: p-value = 0.6286
  Acne_NL vs Acne_L: p-value = 0.4578
P-values for feature:  g__Actinomyces
  Healthy vs Acne_NL: p-valu

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)


P-values for feature:  g__Pasteurella
  Healthy vs Acne_NL: p-value = 0.5563
  Healthy vs Acne_L: p-value = 0.0150
  Acne_NL vs Acne_L: p-value = 0.1886
P-values for feature:  g__Corynebacterium
  Healthy vs Acne_NL: p-value = 0.8626
  Healthy vs Acne_L: p-value = 0.1629
  Acne_NL vs Acne_L: p-value = 0.1378


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:99: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3694974236.py:128: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [272]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
from qiime2 import Artifact

# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)')#,
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]
    else:
        return parts[-1]

# Function to calculate relative abundance
def calculate_relative_abundance(df):
    return df.div(df.sum(axis=1), axis=0)

# Function to apply arcsine square root transformation
def arcsine_transform(df):
    return np.arcsin(np.sqrt(df))

# Function to add significance brackets manually
def add_significance_bracket(ax, start, end, height, text, linewidth=1.0):
    ax.plot([start, end], [height, height], color='black', lw=linewidth)
    ax.text((start + end) * 0.5, height, text, ha='center', va='bottom', fontsize=12)

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    # Normalize the absolute abundance to relative abundance
    relative_abundance_df = calculate_relative_abundance(feature_abundance_df)
    relative_abundance_df += 1e-6  # Add pseudocount to avoid zeros

    # Apply arcsine square root transformation
    arcsine_transformed_df = arcsine_transform(relative_abundance_df)
    
    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])
    arcsine_transformed_df = arcsine_transformed_df.loc[common_samples, :]
    metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
    arcsine_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']

    # Set the explicit order of the Group column
    arcsine_transformed_df['Group'] = pd.Categorical(arcsine_transformed_df['Group'], categories=['Healthy', 'Acne_NL', 'Acne_L'], ordered=True)

    # Load the subject_biplot artifact to get feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract the feature coordinates
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to feature coordinates
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude and select top 9 features
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(9).index

    # Create a figure with 3 rows of subplots
    fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)
    axes = axes.flatten()
# Loop over each top feature and create a single plot for each
for i, feature in enumerate(top_features):
    plt.figure(figsize=(4, 6))  # Define the size for each single plot (smaller and skinnier)
    
    # Merge feature data with metadata
    feature_data = arcsine_transformed_df[[feature, 'Group']].copy()

    # Plot the arcsine-transformed relative abundance as a box plot
    sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors, width=0.4)
    sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)
    
    # Set the y-axis label to the taxon name
    plt.ylabel(f'Arcsine Transformed\n{feature_coordinates.loc[feature, "Taxon"]}', fontsize=12)

    # Perform Mann-Whitney U test for significance
    group_labels = ['Healthy', 'Acne_NL', 'Acne_L']
    pairs = [(0, 1), (0, 2), (1, 2)]
    p_values = []
    print(f"P-values for feature: {feature_coordinates.loc[feature, 'Taxon']}")
    for pair in pairs:
        group1 = group_labels[pair[0]]
        group2 = group_labels[pair[1]]
        data1 = feature_data[feature_data['Group'] == group1][feature]
        data2 = feature_data[feature_data['Group'] == group2][feature]
        _, p_value = mannwhitneyu(data1, data2)
        p_values.append(p_value)
        print(f"  {group1} vs {group2}: p-value = {p_value:.4f}")

    # Set height for significance brackets
    max_height = feature_data[feature].max() * 1.1
    for j, (start, end) in enumerate(pairs):
        height = max_height * (1 + j * 0.1)  # Adjust the height for each bracket
        # Determine the significance label, only add if significant (p < 0.05)
        if p_values[j] < 0.001:
            significance = "***"
        elif p_values[j] < 0.01:
            significance = "**"
        elif p_values[j] < 0.05:
            significance = "*"
        else:
            continue  # Skip adding "ns" for non-significant results

        # Add the significance bracket only for significant comparisons
        add_significance_bracket(plt.gca(), start, end, height, significance)

    # Set title and save the figure for each feature
    #plt.title(f'Top Feature: {feature_coordinates.loc[feature, "Taxon"]}')
    plt.tight_layout()

    # Save each plot as an individual image file
    plt.savefig(f'../Figures/16S_Figures/single_plots_2/{file_suffix}_feature_{i+1}.png', dpi=300)
    
    # Show the plot (optional, if you want to see each one)
    plt.show




Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3184533528.py:98: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  f__Caulobacteraceae
  Healthy vs Acne_NL: p-value = 0.4745
  Healthy vs Acne_L: p-value = 0.0844
  Acne_NL vs Acne_L: p-value = 1.0000


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3184533528.py:98: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  g__Cutibacterium
  Healthy vs Acne_NL: p-value = 0.0340
  Healthy vs Acne_L: p-value = 0.0271
  Acne_NL vs Acne_L: p-value = 0.3961


...

P-values for feature:  f__Neisseriaceae
  Healthy vs Acne_NL: p-value = 0.0935
  Healthy vs Acne_L: p-value = 0.0029
  Acne_NL vs Acne_L: p-value = 0.6085


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3184533528.py:98: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  f__Microbacteriaceae
  Healthy vs Acne_NL: p-value = 0.4745
  Healthy vs Acne_L: p-value = 0.7277
  Acne_NL vs Acne_L: p-value = 0.5517


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3184533528.py:98: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  g__Phenylobacterium
  Healthy vs Acne_NL: p-value = 0.1325
  Healthy vs Acne_L: p-value = 0.0404
  Acne_NL vs Acne_L: p-value = 0.4617


In [266]:
print("Available features in arcsine_transformed_df:", arcsine_transformed_df.columns)
print("Available features in feature_coordinates:", feature_coordinates.index)


Available features in arcsine_transformed_df: Index(['TACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTTAACTCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGG',
       'TACGGAGGGAGCTAGCGTTGTTCGGAATTACTGGGCGTAAAGCGCGCGTAGGCGGTCTTTTAAGTCAGAGGTGAAAGCCCGGGGCTCAACCCCGGAATAGCCTTTGAAACTGGAAGACTTGAATCTTGGAGAGGTCAGTGGAATTCCGAG',
       'TACGTAGGGTACTAGCGTTGTCCGGAATTATTGGGCGTAAAGAGCTCGTAGGTGGTTTGTCGCGTCTGCTGTGGAAACGTGCCGCTTAACGGTGCGCGTGCAGTGGGTACGGGCGGACTAGAGTGCAGTAGGGGAGTCTGGAATTCCTGG',
       'TACGTAGGTGGCAAGCGTTGTCCGGATTTATTGGGCGTAAAGCGCGCGCAGGTGGTTTCTTAAGTCTGATGTGAAATCTCGCGGCTCAACCGCGAGCGGTCATTGGAAACTGGGGAACTTGAGTGCAGAAGAGGATAGTGGAATTCCAAG',
       'TACGAAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCGCGTAGGTGGTTCGTTAAGTTGGATGTGAAAGCCCCGGGCTCAACCTGGGAACTGCATCCAAAACTGGCGAGCTAGAGTATGGCAGAGGGTGGTGGAATTTCCTG',
       'TACGGAGGATGCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGTGCGTAGGCGGCCTGTTAAGTCAGCGGTGAAATCTAGGAGCTTAACTCCTAAATTGCCATTGATACTGGCGGGCTTGAGTGTAGATGAGGGAGGCGG

In [275]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
from qiime2 import Artifact

# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]
    else:
        return parts[-1]

# Function to calculate relative abundance
def calculate_relative_abundance(df):
    return df.div(df.sum(axis=1), axis=0)

# Function to apply arcsine square root transformation
def arcsine_transform(df):
    return np.arcsin(np.sqrt(df))

# Function to add significance brackets manually
def add_significance_bracket(ax, start, end, height, text, linewidth=1.0):
    ax.plot([start, end], [height, height], color='black', lw=linewidth)
    ax.text((start + end) * 0.5, height, text, ha='center', va='bottom', fontsize=12)

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    # Normalize the absolute abundance to relative abundance
    relative_abundance_df = calculate_relative_abundance(feature_abundance_df)
    relative_abundance_df += 1e-6  # Add pseudocount to avoid zeros

    # Apply arcsine square root transformation
    arcsine_transformed_df = arcsine_transform(relative_abundance_df)
    
    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])
    arcsine_transformed_df = arcsine_transformed_df.loc[common_samples, :]
    metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
    arcsine_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']

    # Set the explicit order of the Group column
    arcsine_transformed_df['Group'] = pd.Categorical(arcsine_transformed_df['Group'], categories=['Healthy', 'Acne_NL', 'Acne_L'], ordered=True)

    # Load the subject_biplot artifact to get feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract the feature coordinates
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to feature coordinates
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude and select top 9 features
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(9).index

    # Create a figure with 3 rows of subplots
    fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)
    axes = axes.flatten()

    # Define custom multi-line x-axis labels with sample sizes
    custom_x_labels = [
        'Healthy\nControl\n(n=57)', 
        'Acne\nNon-lesional\n(n=27)', 
        'Acne\nLesional\n(n=159)'
    ]

    # Loop over each top feature and create a single plot for each
    for i, feature in enumerate(top_features):
        plt.figure(figsize=(4, 6))  # Define the size for each single plot (smaller and skinnier)
        
        # Merge feature data with metadata
        feature_data = arcsine_transformed_df[[feature, 'Group']].copy()

        # Plot the arcsine-transformed relative abundance as a box plot
        sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors, width=0.4)
        sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)
        
        # Set the y-axis label to the taxon name
        plt.ylabel(f'Arcsine Transformed\n{feature_coordinates.loc[feature, "Taxon"]}', fontsize=12)

        # Perform Mann-Whitney U test for significance
        group_labels = ['Healthy', 'Acne_NL', 'Acne_L']
        pairs = [(0, 1), (0, 2), (1, 2)]
        p_values = []
        print(f"P-values for feature: {feature_coordinates.loc[feature, 'Taxon']}")
        for pair in pairs:
            group1 = group_labels[pair[0]]
            group2 = group_labels[pair[1]]
            data1 = feature_data[feature_data['Group'] == group1][feature]
            data2 = feature_data[feature_data['Group'] == group2][feature]
            _, p_value = mannwhitneyu(data1, data2)
            p_values.append(p_value)
            print(f"  {group1} vs {group2}: p-value = {p_value:.4f}")

        # Set height for significance brackets
        max_height = feature_data[feature].max() * 1.1
        for j, (start, end) in enumerate(pairs):
            height = max_height * (1 + j * 0.1)  # Adjust the height for each bracket
            # Determine the significance label, only add if significant (p < 0.05)
            if p_values[j] < 0.001:
                significance = "***"
            elif p_values[j] < 0.01:
                significance = "**"
            elif p_values[j] < 0.05:
                significance = "*"
            else:
                continue  # Skip adding "ns" for non-significant results

            # Add the significance bracket only for significant comparisons
            add_significance_bracket(plt.gca(), start, end, height, significance)

        # Set title and save the figure for each feature
        plt.tight_layout()

        # Update x-axis labels and remove "Group" label
        plt.gca().set_xticklabels(custom_x_labels, fontsize=10)
        plt.gca().set_xlabel('')  # Remove the "Group" label

        # Save each plot as an individual image file, including file_suffix for both folders
        plt.savefig(f'../Figures/16S_Figures/single_plots_2/{file_suffix}_feature_{i+1}.png', dpi=300)
        
        # Show the plot (optional, if you want to see each one)
        plt.show()


Processing folder: 174950


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:106: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  f__Caulobacteraceae
  Healthy vs Acne_NL: p-value = 0.4745
  Healthy vs Acne_L: p-value = 0.0844
  Acne_NL vs Acne_L: p-value = 1.0000


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:153: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:106: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  g__Cutibacterium
  Healthy vs Acne_NL: p-value = 0.0340
  Healthy vs Acne_L: p-value = 0.0271
  Acne_NL vs Acne_L: p-value = 0.3961


...

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:153: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:106: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  g__Pasteurella
  Healthy vs Acne_NL: p-value = 0.5563
  Healthy vs Acne_L: p-value = 0.0150
  Acne_NL vs Acne_L: p-value = 0.1886


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:153: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:106: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)


P-values for feature:  g__Corynebacterium
  Healthy vs Acne_NL: p-value = 0.8626
  Healthy vs Acne_L: p-value = 0.1629
  Acne_NL vs Acne_L: p-value = 0.1378


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/613801388.py:153: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# RCLR transformed

In [ ]:
# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    # Convert the biom table's data to a dense array and calculate feature prevalence
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]  # Convert to 1D array using .A1
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
from qiime2 import Artifact

# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]
    else:
        return parts[-1]


# Function to add significance brackets manually
def add_significance_bracket(ax, start, end, height, text, linewidth=1.0):
    ax.plot([start, end], [height, height], color='black', lw=linewidth)
    ax.text((start + end) * 0.5, height, text, ha='center', va='bottom', fontsize=12)

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    feature_abundance_qza = Artifact.load(f'../Data/16S/CTF/filtered_{folder}_T.qza')
    feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    rclr_transformed_df = rclr_transformation(feature_abundance_df)


    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data
    common_samples = arcsine_transformed_df.index.intersection(metadata_df['SampleID'])
    clr_transformed_df = rclr_transformed_df.loc[common_samples, :]
    metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
    rclr_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']

    # Set the explicit order of the Group column
    rclr_transformed_df['Group'] = pd.Categorical(rclr_transformed_df['Group'], categories=['Healthy', 'Acne_NL', 'Acne_L'], ordered=True)

    # Load the subject_biplot artifact to get feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Extract the feature coordinates
    #feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    #feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load taxonomy data
    #taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Add the last two parts of the taxon to feature coordinates
    #feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))

    # Sort feature coordinates by magnitude and select top 9 features
    #top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(9).index

    # Load and filter the BIOM table by prevalence
    biom_path = f'/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Data/16S/RPCA/{folder}_rarefied_table_unannotated_absolute_abundances.biom'
    biom_table = load_table(biom_path)
    prevalence = calculate_prevalence(biom_table)
    prevalent_features = prevalence[prevalence >= 0.1].index  # Minimum 10% prevalence
    
    # Filter feature coordinates by prevalent features
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates = feature_coordinates.loc[feature_coordinates.index.isin(prevalent_features)]
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))
    
    # Select top 10 features by magnitude for plotting
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    top_feature_coords = feature_coordinates.loc[top_features]
        
    
    
    # Create a figure with 3 rows of subplots
    fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)
    axes = axes.flatten()

    # Define custom multi-line x-axis labels with sample sizes
    custom_x_labels = [
        'Healthy\nControl\n(n=57)', 
        'Acne\nNon-lesional\n(n=27)', 
        'Acne\nLesional\n(n=159)'
    ]

    # Loop over each top feature and create a single plot for each
    for i, feature in enumerate(top_features):
        plt.figure(figsize=(4, 6))  # Define the size for each single plot (smaller and skinnier)
        
        # Merge feature data with metadata
        feature_data = rclr_transformed_df[[feature, 'Group']].copy()

        # Plot the arcsine-transformed relative abundance as a box plot
        sns.boxplot(x='Group', y=feature, data=feature_data, palette=group_colors, width=0.4)
        sns.stripplot(x='Group', y=feature, data=feature_data, palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3)
        
        # Set the y-axis label to the taxon name
        plt.ylabel(f'RCLR Transformed\n{feature_coordinates.loc[feature, "Taxon"]}', fontsize=12)

        # Perform Mann-Whitney U test for significance
        group_labels = ['Healthy', 'Acne_NL', 'Acne_L']
        pairs = [(0, 1), (0, 2), (1, 2)]
        p_values = []
        print(f"P-values for feature: {feature_coordinates.loc[feature, 'Taxon']}")
        for pair in pairs:
            group1 = group_labels[pair[0]]
            group2 = group_labels[pair[1]]
            data1 = feature_data[feature_data['Group'] == group1][feature]
            data2 = feature_data[feature_data['Group'] == group2][feature]
            _, p_value = mannwhitneyu(data1, data2)
            p_values.append(p_value)
            print(f"  {group1} vs {group2}: p-value = {p_value:.4f}")

        # Set height for significance brackets
        max_height = feature_data[feature].max() * 1.1
        for j, (start, end) in enumerate(pairs):
            height = max_height * (1 + j * 0.1)  # Adjust the height for each bracket
            # Determine the significance label, only add if significant (p < 0.05)
            if p_values[j] < 0.001:
                significance = "***"
            elif p_values[j] < 0.01:
                significance = "**"
            elif p_values[j] < 0.05:
                significance = "*"
            else:
                continue  # Skip adding "ns" for non-significant results

            # Add the significance bracket only for significant comparisons
            add_significance_bracket(plt.gca(), start, end, height, significance)

        # Set title and save the figure for each feature
        plt.tight_layout()

        # Update x-axis labels and remove "Group" label
        plt.gca().set_xticklabels(custom_x_labels, fontsize=10)
        plt.gca().set_xlabel('')  # Remove the "Group" label

        # Save each plot as an individual image file, including file_suffix for both folders
        plt.savefig(f'../Figures/16S_Figures/rclr_transformed/{file_suffix}_feature_{i+1}.png', dpi=300)
        
        # Show the plot (optional, if you want to see each one)
        plt.show()


## RPCA

In [90]:
# import Python packages
import pandas as pd
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import scipy
import scipy.stats as ss
from skbio.stats.distance import permanova
import biom
from biom import load_table
from gemelli.rpca import rpca #import auto_rpca
from gemelli.utils import qc_rarefaction
from skbio.stats.distance import permanova, DistanceMatrix
from adjustText import adjust_text

In [91]:
# Set the random seed for reproducibility
np.random.seed(123)
# Define the color palette for the groups
group_colors = {
    "Healthy": "#58a35b",  # Color for Healthy
    "Acne_L": "#dd7966",   # Color for Acne Lesional
    "Acne_NL": "#6ab2bd"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom')
]

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Read the metadata file
    metadata_file = '../Metadata/metadata_final_18062024.tsv'
    metadata_df = pd.read_csv(metadata_file, sep='\t')

    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])
    
    # calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "Group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174950 - V1-V3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1111988175.py:73: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

Processing dataset: 174951 - V4


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/1111988175.py:73: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

In [92]:
#look at the severity
# Set the random seed for reproducibility
np.random.seed(123)
# Define the color palette for the groups
local_lesion_severity_colors = {
    "0": "#006f00",
    "1": "#9af09b", # Color for Healthy
    "2": "#b6ddea",   # Color for Acne Lesional
    "3": "#a4a4ff",   # Color for Acne Non-lesional
    "4": "#000096",
    "5": "#ff676c",
    "6": "#960000"
}

# Map the old group names to new ones for the legend
group_name_mapping  = {
    "0":"0",
    "1": "1",  
    "2": "2",   
    "3": "3",   
    "4": "4",
    "5": "5",
    "6": "6"
}

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')

]

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Read the metadata file
    metadata_file = '../Metadata/metadata_final_18062024.tsv'
    metadata_df = pd.read_csv(metadata_file, sep='\t')
    # Ensure local_lesion_severity is treated as a string (categorical variable)
    metadata_df["local_lesion_severity"] = metadata_df["local_lesion_severity"].astype(str)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["local_lesion_severity"] = spca_df.index.map(metadata_df.set_index("SampleID")["local_lesion_severity"])
    
    # calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "local_lesion_severity")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('local_lesion_severity'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=local_lesion_severity_colors[group_name],  # Semi-transparent fill color
            edgecolor=local_lesion_severity_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
# Calculate mean and covariance matrix for the group if more than 2 points
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        if len(group) > 2:

            cov = np.cov(points, rowvar=False)
    
            # Draw the ellipse for the group (with scale factor applied)
            draw_ellipse(mean, cov, ax, local_lesion_severity_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=local_lesion_severity_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_local_lesion_severity_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/937241870.py:90: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=local_lesion_severity_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated sin

Processing dataset: 174950 - V1-V3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/937241870.py:90: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=local_lesion_severity_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated sin

In [122]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_18062024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

# Specify the group column
group_column = 'group' 

# Calculate the count of local_lesion_severity for each group
severity_counts = (
    metadata_df.groupby([group_column, 'local_lesion_severity'])
    .size()
    .unstack(fill_value=0)
)

# Print the result
print(severity_counts)


local_lesion_severity   0   1   2   3   4   5  6
group                                           
Acne_L                  0  38  32  36  28  19  6
Acne_NL                27  18   5   0   0   0  0
Healthy                57   0   0   0   0   0  0


In [123]:
# Create a mapping for severity levels
severity_mapping = {
    0: 'absent',      # You can change 'none' to another term if desired
    1: 'low',
    2: 'low',
    3: 'moderate',
    4: 'moderate',
    5: 'high',
    6: 'high'
}

# Add the severity_level column to metadata_df
metadata_df['severity_level'] = metadata_df['local_lesion_severity'].map(severity_mapping)

# Print the updated metadata DataFrame
print(metadata_df)

              SampleID c_zone  \
0    LAMI.RD308.D16.C1     C1   
1    LAMI.RD310.D21.C1     C1   
2    LAMI.RD305.D21.C3     C3   
3    LAMI.RD306.D18.C2     C2   
4     LAMI.RD306.D7.C2     C2   
..                 ...    ...   
261  LAMI.RD317.D21.C1     C1   
262   LAMI.RD001.D0.C1     C1   
263  LAMI.RD014.D14.C2     C2   
264   LAMI.RD314.D0.C1     C1   
265  LAMI.RD001.D14.C2     C2   

    visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face  \
0                                       not applicable                  
1                                                   72                  
2                                                   69                  
3                                       not applicable                  
4                                                   90                  
..                                                 ...                  
261                                                 77                  
262                

In [124]:
# Create the combined column for severity and group
metadata_df['severity_group'] = metadata_df['severity_level'] + ' ' + metadata_df['group']


In [125]:
# Print unique values in the severity_group column
unique_severity_groups = metadata_df['severity_group'].unique()
print(unique_severity_groups)


['moderate Acne_L' 'absent Acne_NL' 'low Acne_L' 'absent Healthy'
 'low Acne_NL' 'high Acne_L']


In [54]:

# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL":"#9af09b",
    "low Acne_NL": "#b6ddea", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L","high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "severity_group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors and shapes
    for severity_group in severity_order:
        group = spca_df[spca_df['severity_group'] == severity_group]
        for group_name, group_data in group.groupby('group'):
            sns.scatterplot(
                data=group_data,
                x="PC1",
                y="PC2",
                facecolor=severity_group_colors[severity_group],
                edgecolor=severity_group_colors[severity_group],
                alpha=0.8,
                linewidth=0.5,
                ax=ax,
                label=severity_group,
                marker=group_shape_mapping[group_name],  # Use shape mapping for each group
                s=10
            )
            
            # Calculate mean and covariance matrix for the group if more than 2 points
            points = group_data[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            if len(group_data) > 2:
                cov = np.cov(points, rowvar=False)
                draw_ellipse(mean, cov, ax, severity_group_colors[severity_group], scale_factor=2)

            # Add a subtle star (8-pointed) at the mean location for each group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/847953302.py:83: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor rele

Processing dataset: 174950 - V1-V3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/847953302.py:83: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor rele

In [213]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_22102024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

In [214]:
# Now filter out the samples that match 'low Acne_NL'
samples_to_drop = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']

# Get the number of samples that are being dropped
num_dropped_samples = len(samples_to_drop)

# Drop the samples
metadata_df = metadata_df[metadata_df['severity_group'] != 'low Acne_NL']

# Output the number of dropped samples
print(f'Number of samples dropped: {num_dropped_samples}')

Number of samples dropped: 23


In [121]:

# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL":"#9af09b", 
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L","high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom')#,
    #('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]


# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")




Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!


In [108]:
# Verify if the samples to remove are present in the original dataset
samples_not_in_biom = set(samples_to_remove) - set(original_samples)

if samples_not_in_biom:
    print(f"The following samples to remove are not present in the BIOM table: {samples_not_in_biom}")
else:
    print("All samples to remove are present in the BIOM table.")
    
# Verify if the dropped samples match the expected samples
expected_set = set(samples_to_remove)
dropped_set = set(dropped_samples)

if expected_set == dropped_set:
    print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
else:
    print("There is a mismatch in the samples dropped. Please check!")
    
    # Print the differences
    print("Samples expected to be removed but not found in the BIOM table:")
    print(expected_set - dropped_set)
    
    print("Samples dropped from the BIOM table but not expected to be removed:")
    print(dropped_set - expected_set)


The following samples to remove are not present in the BIOM table: {'LAMI.RD317.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD306.D7.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD306.D0.C3'}
There is a mismatch in the samples dropped. Please check!
Samples expected to be removed but not found in the BIOM table:
{'LAMI.RD317.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD306.D7.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD306.D0.C3'}
Samples dropped from the BIOM table but not expected to be removed:
set()


In [127]:
# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L", "high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")
    
    # Use filtered biom table for RPCA
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl_filtered)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "severity_group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors and shapes
    for severity_group in severity_order:
        group = spca_df[spca_df['severity_group'] == severity_group]
        for group_name, group_data in group.groupby('group'):
            sns.scatterplot(
                data=group_data,
                x="PC1",
                y="PC2",
                facecolor=severity_group_colors[severity_group],
                edgecolor=severity_group_colors[severity_group],
                alpha=0.8,
                linewidth=0.5,
                ax=ax,
                label=severity_group,
                marker=group_shape_mapping[group_name],  # Use shape mapping for each group
                s=10
            )
            
            # Calculate mean and covariance matrix for the group if more than 2 points
            points = group_data[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            if len(group_data) > 2:
                cov = np.cov(points, rowvar=False)
                draw_ellipse(mean, cov, ax, severity_group_colors[severity_group], scale_factor=2)

            # Add a subtle star (8-pointed) at the mean location for each group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/437375351.py:115: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width,

Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD310.D7.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD314.D14.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/437375351.py:115: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group],
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2662478306.py:65: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width,

In [129]:
#single elipse
# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#006f00",
    "absent Acne_NL": "#9af09b",
    "low Acne_L": "#ff676c",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_NL", "low Acne_L", "moderate Acne_L", "high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Function to draw an ellipse
def draw_ellipse(mean, cov, ax, color, scale_factor=1):
    from matplotlib.patches import Ellipse
    import matplotlib.transforms as transforms
    import numpy as np

    # Get eigenvalues and eigenvectors of the covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Calculate the angle between the largest eigenvector and the x-axis
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))

    # Width and height are "full" lengths, so scale them by the factor
    width, height = 2 * np.sqrt(eigvals) * scale_factor

    # Draw the ellipse
    ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
    ax.add_patch(ellipse)

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")
    
    # Use filtered biom table for RPCA
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl_filtered)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "severity_group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors and shapes
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:
            group = spca_df[spca_df['severity_group'] == severity_group]
            for group_name, group_data in group.groupby('group'):
                sns.scatterplot(
                    data=group_data,
                    x="PC1",
                    y="PC2",
                    facecolor=severity_group_colors[severity_group],
                    edgecolor=severity_group_colors[severity_group],
                    alpha=0.8,
                    linewidth=0.5,
                    ax=ax,
                    label=severity_group,
                    marker=group_shape_mapping[group_name],  # Use shape mapping for each group
                    s=10
                )

    # Now plot a single ellipse around each of the main groups (Healthy, Acne_L, and Acne_NL)
    for group_name, group_data in spca_df.groupby('group'):
        points = group_data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        if len(group_data) > 2:
            cov = np.cov(points, rowvar=False)
            # Define unique colors for the ellipses around the main groups
            ellipse_color = {
                "Healthy": "#006f00",    # Green for Healthy
                "Acne_NL": "#9af09b",    # Light green for Acne_NL
                "Acne_L": "#ff676c"      # Red for Acne_L
            }[group_name]

            draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)

    # Add star markers for the mean location of each severity group (except dropped 'low Acne_NL')
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:  # Add check to ensure severity group is present
            group = spca_df[spca_df['severity_group'] == severity_group]
            points = group[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            # Add a star marker at the mean location of each severity group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, c

Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD310.D7.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD314.D14.C3'}
There is a mismatch in the samples dropped. Please check!


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/3016173210.py:64: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, c

In [216]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_22102024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

In [217]:
from itertools import combinations
import pandas as pd

# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#423fa6",
    "absent Acne_NL": "#67b2be", 
    "low Acne_L": "#df7963",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_L", "moderate Acne_L", "high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Function to draw an ellipse
def draw_ellipse(mean, cov, ax, color, scale_factor=1):
    from matplotlib.patches import Ellipse
    import matplotlib.transforms as transforms
    import numpy as np

    # Get eigenvalues and eigenvectors of the covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Calculate the angle between the largest eigenvector and the x-axis
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))

    # Width and height are "full" lengths, so scale them by the factor
    width, height = 2 * np.sqrt(eigvals) * scale_factor

    # Draw the ellipse
    ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
    ax.add_patch(ellipse)

# Function to perform pairwise PERMANOVA
def perform_pairwise_permanova(distance, spca_df, groups_column):
    group_pairs = list(combinations(spca_df[groups_column].unique(), 2))
    permanova_results = {}
    
    for group1, group2 in group_pairs:
        # Subset the dataframe for the two groups
        subset = spca_df[spca_df[groups_column].isin([group1, group2])]
        # Filter the distance matrix for this subset
        distance_subset = distance.filter(subset.index)
        # Perform PERMANOVA on the filtered distance matrix for this subset
        result = permanova(distance_subset, subset, groups_column)
        permanova_results[f"{group1} vs {group2}"] = result['p-value']
    
    return permanova_results

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")
    
    # Use filtered biom table for RPCA
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl_filtered)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Perform pairwise PERMANOVA between each pair of groups
    pairwise_permanova_results = perform_pairwise_permanova(distance, spca_df, "severity_group")
    
    # Print the pairwise PERMANOVA results
    print(f"Pairwise PERMANOVA results for {region_title}:")
    for comparison, p_value in pairwise_permanova_results.items():
        print(f"{comparison}: p-value = {p_value:.3f}")

    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors and shapes
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:
            group = spca_df[spca_df['severity_group'] == severity_group]
            for group_name, group_data in group.groupby('group'):
                sns.scatterplot(
                    data=group_data,
                    x="PC1",
                    y="PC2",
                    facecolor=severity_group_colors[severity_group],
                    edgecolor=severity_group_colors[severity_group],
                    alpha=0.8,
                    linewidth=0.5,
                    ax=ax,
                    label=severity_group,
                    marker=group_shape_mapping[group_name],  # Use shape mapping for each group
                    s=10
                )

    # Now plot a single ellipse around each of the main groups (Healthy, Acne_L, and Acne_NL)
    for group_name, group_data in spca_df.groupby('group'):
        points = group_data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        if len(group_data) > 2:
            cov = np.cov(points, rowvar=False)
            # Define unique colors for the ellipses around the main groups
            ellipse_color = {
                "Healthy": "#423fa6",
                "Acne_NL": "#67b2be",   
                "Acne_L": "#df7963"     
            }[group_name]

            draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)

    # Add star markers for the mean location of each severity group (except dropped 'low Acne_NL')
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:  # Add check to ensure severity group is present
            group = spca_df[spca_df['severity_group'] == severity_group]
            points = group[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            # Add a star marker at the mean location of each severity group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()

    
   


Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!
Pairwise PERMANOVA results for V4:
absent Healthy vs moderate Acne_L: p-value = 0.013
absent Healthy vs low Acne_L: p-value = 0.001
absent Healthy vs absent Acne_NL: p-value = 0.006
absent Healthy vs high Acne_L: p-value = 0.002
moderate Acne_L vs low Acne_L: p-value = 0.239
moderate Acne_L vs absent Acne_NL: p-value = 0.890
moderate Acne_L vs high Acne_L: p-value = 0.690
low Acne_L vs absent Acne_NL: p-value = 0.428
low Acne_

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2956183917.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2956183917.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2956183917.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, c

Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD310.D7.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD314.D14.C3'}
There is a mismatch in the samples dropped. Please check!
Pairwise PERMANOVA results for V1-V3:
absent Healthy vs moderate Acne_L: p-value = 0.001
absent Healthy vs absent Acne_NL: p-value = 0.002
absent Healthy vs low Acne_L: p-value = 0.001
absent Healthy vs high Acne_L: p-value = 0.001
moderate Acne_L vs absent Acne_NL: p-value = 0.789
moderate Acne_L vs low Acne_L: p-value = 0.622
moderate Acne_L vs high Acne_L: p-

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2956183917.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2956183917.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/2956183917.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, c

RPCA with features

In [235]:
# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]  # Provide more context if 'uncultured'
    else:
        return parts[-1]  # Use only the last part if no 'uncultured'


In [249]:
from itertools import combinations
import pandas as pd

# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
severity_group_colors = {
    "absent Healthy": "#423fa6",
    "absent Acne_NL": "#67b2be", 
    "low Acne_L": "#df7963",
    "moderate Acne_L": "#ca0000", 
    "high Acne_L": "#960000"
}

# Define the shape for each group
group_shape_mapping = {
    "Healthy": "o",    # Circle
    "Acne_NL": "^",    # Triangle
    "Acne_L": "D",     # Star
}

# Define the order for the legend
severity_order = ["absent Healthy", "absent Acne_NL", "low Acne_L", "moderate Acne_L", "high Acne_L"]

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

# Function to draw an ellipse
def draw_ellipse(mean, cov, ax, color, scale_factor=1):
    from matplotlib.patches import Ellipse
    import matplotlib.transforms as transforms
    import numpy as np

    # Get eigenvalues and eigenvectors of the covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Calculate the angle between the largest eigenvector and the x-axis
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))

    # Width and height are "full" lengths, so scale them by the factor
    width, height = 2 * np.sqrt(eigvals) * scale_factor

    # Draw the ellipse
    ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
    ax.add_patch(ellipse)

# Function to perform pairwise PERMANOVA
def perform_pairwise_permanova(distance, spca_df, groups_column):
    group_pairs = list(combinations(spca_df[groups_column].unique(), 2))
    permanova_results = {}
    
    for group1, group2 in group_pairs:
        # Subset the dataframe for the two groups
        subset = spca_df[spca_df[groups_column].isin([group1, group2])]
        # Filter the distance matrix for this subset
        distance_subset = distance.filter(subset.index)
        # Perform PERMANOVA on the filtered distance matrix for this subset
        result = permanova(distance_subset, subset, groups_column)
        permanova_results[f"{group1} vs {group2}"] = result['p-value']
    
    return permanova_results

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
    
    # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")
    
    # Verify if the dropped samples match the expected samples
    if set(samples_to_remove) == dropped_samples:
        print(f"Correct samples were dropped: {len(dropped_samples)} samples removed.")
    else:
        print("There is a mismatch in the samples dropped. Please check!")
    
    # Use filtered biom table for RPCA
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl_filtered)

    # Clear the figure to avoid overlapping plots
    plt.clf()
    
    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["severity_group"] = spca_df.index.map(metadata_df.set_index("SampleID")["severity_group"])
    spca_df["group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])  # Add group mapping
    
    # Extract the feature ordinations
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)

    # Load the taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)

    # Extract relevant taxonomy information
    taxonomy["short_taxa"] = taxonomy["Taxon"].apply(extract_last_two_taxa)

    # Extract top 10 features based on variance in ordination space
    top_features = feature_coordinates.var(axis=1).sort_values(ascending=False).head(10)
    top_features_df = feature_coordinates.loc[top_features.index]

    # Merge the top features with their taxonomy for labeling
    top_features_with_taxa = top_features_df.merge(taxonomy[['short_taxa']], left_index=True, right_index=True)

    # Perform pairwise PERMANOVA between each pair of groups
    pairwise_permanova_results = perform_pairwise_permanova(distance, spca_df, "severity_group")
    
    # Print the pairwise PERMANOVA results
    print(f"Pairwise PERMANOVA results for {region_title}:")
    for comparison, p_value in pairwise_permanova_results.items():
        print(f"{comparison}: p-value = {p_value:.3f}")

    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Scatter plot of sample ordinations
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:
            group = spca_df[spca_df['severity_group'] == severity_group]
            for group_name, group_data in group.groupby('group'):
                sns.scatterplot(
                    data=group_data,
                    x="PC1",
                    y="PC2",
                    facecolor=severity_group_colors[severity_group],
                    edgecolor=severity_group_colors[severity_group],
                    alpha=0.8,
                    linewidth=0.5,
                    ax=ax,
                    label=severity_group,
                    marker=group_shape_mapping[group_name],
                    s=10
                )

    scaling_factor = 0.5
    # Plot feature vectors (biplot)
    for i, feature in top_features_with_taxa.iterrows():

        # Scale the feature coordinates by the scaling factor
        scaled_pc1 = feature["PC1"] * scaling_factor
        scaled_pc2 = feature["PC2"] * scaling_factor
    
        # Plot feature as arrow from the origin to its location
        ax.arrow(0, 0, scaled_pc1, scaled_pc2, color="grey", head_width=0.01, head_length=0.01, alpha=0.6)
        # Add the corresponding taxonomy label
        ax.text(scaled_pc1 * 1.1, scaled_pc2 * 1.1, feature["short_taxa"], fontsize=5, ha="left")


    # Now plot a single ellipse around each of the main groups (Healthy, Acne_L, and Acne_NL)
    for group_name, group_data in spca_df.groupby('group'):
        points = group_data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        if len(group_data) > 2:
            cov = np.cov(points, rowvar=False)
            # Define unique colors for the ellipses around the main groups
            ellipse_color = {
                "Healthy": "#423fa6",
                "Acne_NL": "#67b2be",   
                "Acne_L": "#df7963"     
            }[group_name]

            draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)

    # Add star markers for the mean location of each severity group (except dropped 'low Acne_NL')
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:  # Add check to ensure severity group is present
            group = spca_df[spca_df['severity_group'] == severity_group]
            points = group[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            
            # Add a star marker at the mean location of each severity group
            ax.scatter(mean[0], mean[1], color=severity_group_colors[severity_group], 
                       marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.25, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='center', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_severity_group_RPCA.png", dpi=600)
    plt.show()

    
   


Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD304.D7.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3'}
There is a mismatch in the samples dropped. Please check!
Pairwise PERMANOVA results for V4:
absent Healthy vs moderate Acne_L: p-value = 0.013
absent Healthy vs low Acne_L: p-value = 0.001
absent Healthy vs absent Acne_NL: p-value = 0.006
absent Healthy vs high Acne_L: p-value = 0.002
moderate Acne_L vs low Acne_L: p-value = 0.239
moderate Acne_L vs absent Acne_NL: p-value = 0.890
moderate Acne_L vs high Acne_L: p-value = 0.690
low Acne_L vs absent Acne_NL: p-value = 0.428
low Acne_

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, colo

Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD310.D7.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD314.D14.C3'}
There is a mismatch in the samples dropped. Please check!
Pairwise PERMANOVA results for V1-V3:
absent Healthy vs moderate Acne_L: p-value = 0.001
absent Healthy vs absent Acne_NL: p-value = 0.002
absent Healthy vs low Acne_L: p-value = 0.001
absent Healthy vs high Acne_L: p-value = 0.001
moderate Acne_L vs absent Acne_NL: p-value = 0.789
moderate Acne_L vs low Acne_L: p-value = 0.622
moderate Acne_L vs high Acne_L: p-

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, color=color, alpha=0.3)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_52625/298337937.py:66: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ellipse = Ellipse(mean, width, height, angle, colo